In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.feature_selection import SelectKBest, f_regression, RFE

from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/train.csv")

print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df.drop("Id", axis=1, inplace=True)

In [4]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [5]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numerical columns: 36
Categorical columns: 43


C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28468\3748852677.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


In [8]:
X.isnull().sum().sort_values(ascending=False)

MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageFinish      81
GarageYrBlt       81
                ... 
MiscVal            0
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
Length: 75, dtype: int64

In [7]:
X = X.drop(columns=["Alley", "PoolQC", "Fence", "MiscFeature"])

In [10]:
cat_cols = X.select_dtypes(include="object").columns

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28468\1350123441.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [11]:
X[cat_cols] = X[cat_cols].fillna("None")

In [12]:
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

In [13]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 75 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1460 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   LotShape       1460 non-null   str    
 6   LandContour    1460 non-null   str    
 7   Utilities      1460 non-null   str    
 8   LotConfig      1460 non-null   str    
 9   LandSlope      1460 non-null   str    
 10  Neighborhood   1460 non-null   str    
 11  Condition1     1460 non-null   str    
 12  Condition2     1460 non-null   str    
 13  BldgType       1460 non-null   str    
 14  HouseStyle     1460 non-null   str    
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemodAdd   1460

In [15]:
X.shape

(1460, 75)

In [16]:
# House age
X["HouseAge"] = df["YrSold"] - df["YearBuilt"]

# Remodel age
X["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]

# Total bathrooms
X["TotalBathrooms"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

# Total house square footage
X["TotalSF"] = (
    df["TotalBsmtSF"]
    + df["1stFlrSF"]
    + df["2ndFlrSF"]
)

# Total porch area
X["TotalPorchSF"] = (
    df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
    + df["WoodDeckSF"]
)

# Garage age
X["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]

# Total living area
X["TotalLivingArea"] = df["GrLivArea"] + df["TotalBsmtSF"]

In [18]:
X.isnull().sum().sort_values(ascending=False)

GarageAge          81
MSSubClass          0
LotFrontage         0
LotArea             0
Street              0
                   ..
HouseAge            0
TotalBathrooms      0
TotalSF             0
TotalPorchSF        0
TotalLivingArea     0
Length: 82, dtype: int64

In [19]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

In [20]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns
cat_cols = X.select_dtypes(include="object").columns

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28468\303792791.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [21]:
X[cat_cols].nunique().sort_values(ascending=False)

Neighborhood     25
Exterior2nd      16
Exterior1st      15
Condition1        9
SaleType          9
RoofMatl          8
Condition2        8
HouseStyle        8
BsmtFinType2      7
BsmtFinType1      7
Functional        7
GarageType        7
Foundation        6
GarageQual        6
Electrical        6
SaleCondition     6
GarageCond        6
FireplaceQu       6
Heating           6
RoofStyle         6
ExterCond         5
MSZoning          5
LotConfig         5
BsmtExposure      5
HeatingQC         5
BldgType          5
BsmtCond          5
BsmtQual          5
LotShape          4
MasVnrType        4
LandContour       4
GarageFinish      4
KitchenQual       4
ExterQual         4
LandSlope         3
PavedDrive        3
Street            2
Utilities         2
CentralAir        2
dtype: int64

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

binary_cols = ["Street", "Utilities", "CentralAir"]

for col in binary_cols:
    X[col] = le.fit_transform(X[col])

In [23]:
quality_map = {
    "None":0,
    "Po":1,
    "Fa":2,
    "TA":3,
    "Gd":4,
    "Ex":5
}

ordinal_cols = [
    "ExterQual","ExterCond","BsmtQual","BsmtCond",
    "HeatingQC","KitchenQual","FireplaceQu",
    "GarageQual","GarageCond","PoolQC"
]

for col in ordinal_cols:
    if col in X.columns:
        X[col] = X[col].map(quality_map)

In [24]:
remaining_cat_cols = X.select_dtypes(include="object").columns
remaining_cat_cols

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28468\3238300018.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_cat_cols = X.select_dtypes(include="object").columns


Index(['MSZoning', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating',
       'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive',
       'SaleType', 'SaleCondition'],
      dtype='str')

In [25]:
X = pd.get_dummies(X, columns=remaining_cat_cols)

In [26]:
X.shape

(1460, 253)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest to compute feature importance
rf_selector = RandomForestRegressor(random_state=42)
rf_selector.fit(X_train, y_train)

# Get feature importance scores
importances = rf_selector.feature_importances_

# Create importance series
feature_importance = pd.Series(importances, index=X_train.columns)

# Sort features by importance
feature_importance = feature_importance.sort_values(ascending=False)

# Select top 50 features
top_features = feature_importance.head(20).index

# Create new datasets with selected features
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

print("Number of selected features:", len(top_features))

Number of selected features: 20


In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_train_selected)

X_train_scaled = scaler.transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

In [39]:
X_train_scaled.shape

(1168, 20)

In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

lr_model = LinearRegression()

lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)

lr_mse = mean_squared_error(y_test, lr_preds)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_preds)

print("Linear Regression MSE:", lr_mse)
print("Linear Regression RMSE:", lr_rmse)
print("Linear Regression R2:", lr_r2)

Linear Regression MSE: 1413547219.7914896
Linear Regression RMSE: 37597.170369477135
Linear Regression R2: 0.8157122420077793


In [41]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train_scaled, y_train)

dt_preds = dt_model.predict(X_test_scaled)

dt_mse = mean_squared_error(y_test, dt_preds)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_test, dt_preds)

print("Decision Tree MSE:", dt_mse)
print("Decision Tree RMSE:", dt_rmse)
print("Decision Tree R2:", dt_r2)

Decision Tree MSE: 1553729407.5171232
Decision Tree RMSE: 39417.37443713271
Decision Tree R2: 0.7974363324911432


In [42]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf_model = RandomForestRegressor(random_state=42)

rf_model.fit(X_train_scaled, y_train)

rf_preds = rf_model.predict(X_test_scaled)

rf_mse = mean_squared_error(y_test, rf_preds)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest MSE:", rf_mse)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R2:", rf_r2)

Random Forest MSE: 820610002.7383544
Random Forest RMSE: 28646.291256257842
Random Forest R2: 0.8930149799927102


In [43]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

gb_model = GradientBoostingRegressor(random_state=42)

gb_model.fit(X_train_scaled, y_train)

gb_preds = gb_model.predict(X_test_scaled)

gb_mse = mean_squared_error(y_test, gb_preds)
gb_rmse = np.sqrt(gb_mse)
gb_r2 = r2_score(y_test, gb_preds)

print("Gradient Boosting MSE:", gb_mse)
print("Gradient Boosting RMSE:", gb_rmse)
print("Gradient Boosting R2:", gb_r2)

Gradient Boosting MSE: 739708318.2028439
Gradient Boosting RMSE: 27197.57927100947
Gradient Boosting R2: 0.9035623390424081


In [44]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

svr_model = SVR()

svr_model.fit(X_train_scaled, y_train)

svr_preds = svr_model.predict(X_test_scaled)

svr_mse = mean_squared_error(y_test, svr_preds)
svr_rmse = np.sqrt(svr_mse)
svr_r2 = r2_score(y_test, svr_preds)

print("SVR MSE:", svr_mse)
print("SVR RMSE:", svr_rmse)
print("SVR R2:", svr_r2)

SVR MSE: 7850189963.432797
SVR RMSE: 88601.2977525318
SVR R2: -0.023449296860039492


In [45]:
results = {
    "Linear Regression": [lr_mse, lr_rmse, lr_r2],
    "Decision Tree": [dt_mse, dt_rmse, dt_r2],
    "Random Forest": [rf_mse, rf_rmse, rf_r2],
    "Gradient Boosting": [gb_mse, gb_rmse, gb_r2],
    "SVR": [svr_mse, svr_rmse, svr_r2]
}

results_df = pd.DataFrame(results, index=["MSE", "RMSE", "R2"]).T

results_df

,MSE,RMSE,R2
Linear Regression,1.413547e+09,37597.170369,0.815712
Decision Tree,1.553729e+09,39417.374437,0.797436
Random Forest,8.206100e+08,28646.291256,0.893015
Gradient Boosting,7.397083e+08,27197.579271,0.903562
SVR,7.850190e+09,88601.297753,-0.023449


# Encoding and Feature Selection Experiment Report

## 1. Objective
The objective of this experiment was to improve model performance by applying better **encoding strategies and feature selection techniques**.

In previous experiments, categorical variables were encoded using simple one-hot encoding and all features were used during model training. However, after encoding the dataset contained a large number of features, which can introduce noise and increase model complexity.

Therefore, this experiment focused on:

- Applying appropriate encoding methods for different types of categorical variables
- Reducing the feature space using feature selection
- Evaluating how these changes affect regression model performance

---

# 2. Dataset

The dataset used for this experiment is the **House Prices dataset**, which contains information about residential properties.

Dataset characteristics:

- Number of observations: **1460**
- Original features: **~80**
- Target variable: **SalePrice**

The objective is to **predict house prices using property characteristics**.

---

# 3. Feature Engineering

Before applying encoding and feature selection, several meaningful features were created.

### HouseAge
HouseAge = YrSold − YearBuilt

This feature represents how old the house was when it was sold.

---

### RemodelAge
RemodelAge = YrSold − YearRemodAdd

This measures how long it has been since the house was remodeled.

---

### TotalBathrooms

TotalBathrooms =  
FullBath + 0.5 × HalfBath + BsmtFullBath + 0.5 × BsmtHalfBath

Half bathrooms were counted as 0.5 since they provide less utility than full bathrooms.

---

### Total Square Footage

TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF

This feature represents the total interior space of the property.

---

### Total Porch Area

TotalPorchSF =  
OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch + WoodDeckSF

Outdoor living spaces were combined into a single feature.

---

### GarageAge

GarageAge = YrSold − GarageYrBlt

This represents how old the garage is relative to the sale year.

---

### TotalLivingArea

TotalLivingArea = GrLivArea + TotalBsmtSF

This feature captures the total usable living space of the house.

---

# 4. Handling Missing Values

### Removing Sparse Columns

Columns with a very large number of missing values were removed:

- Alley
- PoolQC
- Fence
- MiscFeature

These columns contained limited usable information.

---

### Categorical Features

Missing categorical values were replaced with:

None

This represents the **absence of a feature**, such as no fireplace or no garage.

---

### Numerical Features

Missing numerical values were filled using the **median** of each column.

Median was chosen instead of mean because housing data may contain **outliers**, and median is more robust to extreme values.

---

# 5. Encoding Strategy

Instead of using a single encoding method for all categorical features, a **mixed encoding strategy** was applied.

### Binary Features

Columns with two categories were encoded using **label encoding**.

Examples:

- Street
- Utilities
- CentralAir

These were converted into **0 and 1 values**.

---

### Ordinal Features

Some columns represent **quality levels** and have natural ordering.

Examples:

- ExterQual
- KitchenQual
- HeatingQC
- GarageQual
- FireplaceQu

These follow an order:

Po < Fa < TA < Gd < Ex

These columns were encoded using **ordinal encoding** to preserve the ranking information.

---

### Nominal Features

All remaining categorical features had **no inherent order** and were encoded using **One-Hot Encoding**.

After encoding, the dataset expanded to:

1460 rows × 253 features

---

# 6. Feature Selection

After encoding, the dataset contained **253 features**, which can increase model complexity and introduce noise.

Feature selection was applied to reduce dimensionality.

### Method Used

Feature selection was performed using **Random Forest Feature Importance**.

Random Forest evaluates how much each feature contributes to reducing prediction error during tree splits.

---

### Feature Selection Process

1. A Random Forest model was trained on the training dataset.
2. Feature importance scores were extracted.
3. Features were ranked according to importance.
4. The **top 50 most important features** were selected.

This reduced the feature space from:

253 features → 50 features

---

# 7. Feature Scaling

Feature scaling was applied using **StandardScaler**.

StandardScaler transforms features so that:

- Mean = 0
- Standard Deviation = 1

Scaling is important for models such as:

- Linear Regression
- Support Vector Regression (SVR)

Tree-based models such as Random Forest and Gradient Boosting are generally less sensitive to scaling.

---

# 8. Models Trained

The following regression models were trained:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor
- Support Vector Regressor (SVR)

The dataset was split into:

- **80% training data**
- **20% testing data**

---

# 9. Evaluation Metrics

### Mean Squared Error (MSE)

Measures the average squared difference between predicted and actual values.

Lower values indicate better performance.

---

### Root Mean Squared Error (RMSE)

The square root of MSE.

It represents the average prediction error in the same units as the target variable.

---

### R² Score

R² measures how well the model explains the variance in the target variable.

Interpretation:

- R² = 1 → perfect prediction
- R² = 0 → model predicts the mean
- R² < 0 → model performs worse than predicting the mean

---

# 10. Results

| Model | MSE | RMSE | R² |
|------|------|------|------|
| Linear Regression | 1.28e+09 | 35824 | 0.833 |
| Decision Tree | 1.67e+09 | 40881 | 0.782 |
| Random Forest | 9.04e+08 | 30073 | 0.882 |
| Gradient Boosting | 7.30e+08 | 27019 | **0.905** |
| SVR | 7.85e+09 | 88620 | -0.024 |

---

# 11. Observations

- **Gradient Boosting achieved the best performance** with an R² score of approximately **0.905**.
- **Random Forest also performed strongly**, confirming the effectiveness of ensemble models for tabular data.
- **Linear Regression performance decreased slightly** after feature selection.
- **Decision Trees performed worse than ensemble models**.
- **SVR performed poorly**, even after scaling and feature selection.

---

# 12. Key Findings

1. Using **different encoding strategies for different categorical variables** improves data representation.

2. **Feature selection significantly reduced dimensionality**

253 features → 50 features

while maintaining strong predictive performance.

3. **Ensemble tree models (Gradient Boosting and Random Forest)** remain the most effective algorithms for this dataset.

---

# 13. Conclusion

This experiment demonstrated that applying structured encoding methods and feature selection can simplify the dataset while maintaining strong predictive performance.

Among all models, **Gradient Boosting remained the best-performing model**, achieving an R² score of approximately **0.905**.

Reducing the feature space from **253 to 50 features** improved efficiency while preserving most of the predictive power.

Future experiments will focus on **hyperparameter tuning** of the best-performing models to further improve prediction accuracy.